# Early Cancer Signal Detector — EDA

This notebook explores the NHANES dataset after processing.

**Contents:**
1. Dataset overview & class balance
2. Lab value distributions (cancer vs control)
3. NLR, PLR, SII analysis
4. Correlation heatmap
5. Feature importance preview

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

PROCESSED = Path('../data/processed')

In [ ]:
# Load model dataset
df = pd.read_parquet(PROCESSED / 'model_dataset.parquet')
print(f'Total participants: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nCancer prevalence:')
print(df[['any_cancer', 'cancer_colorectal', 'cancer_lung', 'cancer_breast',
          'cancer_prostate', 'cancer_leukemia', 'cancer_lymphoma']].mean().mul(100).round(2).to_string())

In [ ]:
# Run feature engineering
from src.features.lab_features import build_feature_matrix
df_feat, feature_cols = build_feature_matrix(df)
print(f'Feature matrix: {df_feat.shape}')
print(f'Features: {len(feature_cols)}')

## 1. Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall cancer flag
vc = df['any_cancer'].value_counts()
axes[0].pie(vc, labels=['No Cancer', 'Cancer'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Overall Cancer Label')

# By subtype
subtypes = ['cancer_colorectal', 'cancer_lung', 'cancer_breast',
            'cancer_prostate', 'cancer_leukemia', 'cancer_lymphoma', 'cancer_pancreatic']
subtype_counts = {
    c.replace('cancer_', '').title(): df[c].sum()
    for c in subtypes if c in df.columns
}
axes[1].barh(list(subtype_counts.keys()), list(subtype_counts.values()), color='steelblue')
axes[1].set_title('Cancer Cases by Subtype')
axes[1].set_xlabel('N cases')

plt.tight_layout()
plt.show()

## 2. Key Lab Distributions: Cancer vs Control

In [ ]:
key_labs = ['hgb', 'plt', 'albumin', 'crp', 'nlr', 'ldh', 'ferritin', 'wbc']
existing = [l for l in key_labs if l in df_feat.columns]

cancer_group   = df_feat[df_feat['any_cancer'] == 1]
control_group  = df_feat[df_feat['any_cancer'] == 0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, lab in enumerate(existing[:8]):
    ax = axes[i]
    c_vals = cancer_group[lab].dropna()
    n_vals = control_group[lab].dropna()
    
    # Clip to 99th percentile for visualization
    upper = np.percentile(df_feat[lab].dropna(), 99)
    lower = np.percentile(df_feat[lab].dropna(), 1)
    
    ax.hist(n_vals.clip(lower, upper), bins=40, alpha=0.5, label='No Cancer',
            color='#4CAF50', density=True)
    ax.hist(c_vals.clip(lower, upper), bins=40, alpha=0.6, label='Cancer',
            color='#F44336', density=True)
    ax.set_title(lab.upper(), fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlabel('')
    
    # Mann-Whitney U test
    from scipy import stats
    if len(c_vals) > 5 and len(n_vals) > 5:
        stat, p = stats.mannwhitneyu(c_vals, n_vals, alternative='two-sided')
        ax.set_xlabel(f'p={p:.2e}', fontsize=9, color='gray')

plt.suptitle('Lab Value Distributions: Cancer vs Control', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Inflammatory Indices: NLR, PLR, SII

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title in zip(
    axes,
    ['nlr', 'plr', 'log_sii'],
    ['NLR (Neutrophil-to-Lymphocyte)', 'PLR (Platelet-to-Lymphocyte)', 'log(SII)']
):
    if col not in df_feat.columns:
        continue
    data = [
        df_feat[df_feat['any_cancer'] == 0][col].dropna().clip(0, 20),
        df_feat[df_feat['any_cancer'] == 1][col].dropna().clip(0, 20),
    ]
    ax.boxplot(data, labels=['No Cancer', 'Cancer'],
               patch_artist=True,
               boxprops=dict(facecolor='lightyellow'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(col)
    
    # Statistical test
    from scipy import stats
    stat, p = stats.mannwhitneyu(data[0], data[1])
    ax.set_xlabel(f'Mann-Whitney p={p:.2e}')

plt.suptitle('Inflammatory Index Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap (Key Features)

In [ ]:
corr_features = [
    'any_cancer', 'age', 'hgb', 'plt', 'wbc', 'albumin', 'crp',
    'nlr', 'plr', 'sii', 'ldh', 'ferritin', 'egfr', 'bilirubin',
    'neut_pct', 'lymph_pct', 'mcv', 'rdw'
]
existing = [c for c in corr_features if c in df_feat.columns]

corr = df_feat[existing].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-0.5, vmax=0.5,
    annot_kws={'size': 8},
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Age Distribution by Cancer Status

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, color, group in [
    ('No Cancer', '#4CAF50', df_feat[df_feat['any_cancer'] == 0]),
    ('Cancer', '#F44336', df_feat[df_feat['any_cancer'] == 1]),
]:
    ax.hist(group['age'].dropna(), bins=30, alpha=0.6, label=label,
            color=color, density=True, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Age at Exam')
ax.set_ylabel('Density')
ax.set_title('Age Distribution: Cancer vs Control')
ax.legend()
ax.axvline(50, color='gray', linestyle='--', alpha=0.5, label='Age 50')
plt.tight_layout()
plt.show()

# Cancer rate by age decade
df_feat['age_decade'] = (df_feat['age'] // 10 * 10).astype(int)
rate_by_age = df_feat.groupby('age_decade')['any_cancer'].mean() * 100
print('\nCancer rate by age decade:')
print(rate_by_age.to_string())

## 6. Glasgow Prognostic Score Distribution

In [ ]:
if 'gps' in df_feat.columns:
    gps_cancer_rate = df_feat.groupby('gps')['any_cancer'].mean() * 100
    gps_counts = df_feat['gps'].value_counts().sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(gps_counts.index, gps_counts.values, color=['#4CAF50', '#FF9800', '#F44336'])
    axes[0].set_title('GPS Distribution')
    axes[0].set_xlabel('Glasgow Prognostic Score (0=best, 2=worst)')
    axes[0].set_ylabel('N participants')
    
    axes[1].bar(gps_cancer_rate.index, gps_cancer_rate.values,
                color=['#4CAF50', '#FF9800', '#F44336'])
    axes[1].set_title('Cancer Rate by GPS')
    axes[1].set_xlabel('Glasgow Prognostic Score')
    axes[1].set_ylabel('Cancer rate (%)')
    
    for ax in axes:
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['GPS 0\n(Normal)', 'GPS 1\n(Moderate)', 'GPS 2\n(High inflammation)'])
    
    plt.suptitle('Glasgow Prognostic Score vs Cancer Risk', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Missing Data Profile

In [ ]:
key_features = [
    'wbc', 'hgb', 'plt', 'neut_pct', 'lymph_pct', 'albumin', 'alt', 'ast',
    'creatinine', 'calcium', 'ferritin', 'crp', 'ldh'
]
existing = [f for f in key_features if f in df_feat.columns]
missing_pct = df_feat[existing].isna().mean() * 100

plt.figure(figsize=(10, 4))
colors = ['#dc3545' if v > 30 else '#fd7e14' if v > 10 else '#198754'
          for v in missing_pct.values]
plt.barh(missing_pct.index, missing_pct.values, color=colors)
plt.axvline(30, color='red', linestyle='--', alpha=0.5, label='30% threshold')
plt.xlabel('Missing (%)')
plt.title('Missing Data Rate by Lab')
plt.legend()
plt.tight_layout()
plt.show()